In [3]:
! pip install pandas numpy matplotlib duckdb pyarrow wordcloud scikit-learn



[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
from pathlib import Path
import pandas as pd
import numpy as np

DATA_PATH = Path('data/commentaires_assurance_auto.csv')
OUT_DIR = Path('outputs')
OUT_DIR.mkdir(parents=True, exist_ok=True)

DATA_PATH, OUT_DIR


(WindowsPath('data/commentaires_assurance_auto.csv'), WindowsPath('outputs'))

In [8]:
COLS = [
    "id", "date", "device", "canal", "sentiment", "motif", "contexte",
    "anciennete", "formule", "note", "urgence", "region", "support", "commentaire"
]

df = pd.read_csv(DATA_PATH, sep=";", header=None, names=COLS, encoding="utf-8")
df.head(3)


,id,date,device,canal,sentiment,motif,contexte,anciennete,formule,note,urgence,region,support,commentaire
0,1,2024-11-23,Android,Web,neg,photo,collision,3-10 ans,tous_risques,2,faible,Île-de-France,App,"Très mécontent, les photos restent en attente ..."
1,2,2024-03-22,iOS,App,neg,complexité,collision,0-1 an,tiers,2,moyenne,Pays de la Loire,Téléphone,"Je suis déçu, c'est trop technique pour un sin..."
2,3,2024-04-26,iOS,Email,pos,authentification,collision,3-10 ans,jeune_conducteur,4,faible,Auvergne-Rhône-Alpes,App,"Très satisfait, ça bloque sur l'authentificati..."


In [9]:
print('lignes:', len(df))
print('colonnes:', len(df.columns))
df.info()


lignes: 8650
colonnes: 14
<class 'pandas.DataFrame'>
RangeIndex: 8650 entries, 0 to 8649
Data columns (total 14 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   id           8650 non-null   int64
 1   date         8650 non-null   str  
 2   device       8650 non-null   str  
 3   canal        8650 non-null   str  
 4   sentiment    8650 non-null   str  
 5   motif        8650 non-null   str  
 6   contexte     8650 non-null   str  
 7   anciennete   8650 non-null   str  
 8   formule      8650 non-null   str  
 9   note         8650 non-null   int64
 10  urgence      8650 non-null   str  
 11  region       8650 non-null   str  
 12  support      8650 non-null   str  
 13  commentaire  8650 non-null   str  
dtypes: int64(2), str(12)
memory usage: 2.7 MB


In [10]:
expected = {
    'id','date','device','canal','support','sentiment','motif','contexte','region',
    'anciennete','formule','note','urgence','commentaire'
}
missing = sorted(list(expected - set(df.columns)))
extra = sorted(list(set(df.columns) - expected))
print('colonnes manquantes:', missing)
print('colonnes en plus:', extra)


colonnes manquantes: []
colonnes en plus: []


In [11]:
if 'date' in df.columns:
    df['date'] = pd.to_datetime(df['date'], errors='coerce')
for c in ['note','urgence']:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors='coerce')

empty_comments = (df['commentaire'].astype(str).str.strip() == '').sum()
print('commentaires vides:', empty_comments)
df[['date','note','urgence']].describe(include='all')


commentaires vides: 0


,date,note,urgence
count,8650,8650.000000,0.0
mean,2024-07-01 08:12:45.780346,2.315723,NaN
min,2024-01-01 00:00:00,1.000000,NaN
25%,2024-03-31 00:00:00,1.000000,NaN
50%,2024-07-02 00:00:00,2.000000,NaN
75%,2024-10-01 00:00:00,3.000000,NaN
max,2024-12-31 00:00:00,5.000000,NaN
std,NaN,1.182735,NaN


In [12]:
clean_path = OUT_DIR / 'comments_clean.parquet'
df.to_parquet(clean_path, index=False)
clean_path


WindowsPath('outputs/comments_clean.parquet')